# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shashank007-ux/Week-1-Run-the-Starter-Notebooks/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


## 1. My rule and its reason codes

A page receives a higher review score when it has meaningful search visibility, has not been updated recently, has an observable ranking or click-through opportunity, or is thin relative to visible demand. The score uses only current/trailing-window measurements and content metadata; it does not use `trend_direction` or `trend_pct`, which define the evaluation label.

Reason codes explain the review queue:

- `stale_visible_page`: at least 180 days since update and at least 500 impressions.
- `low_ctr_visible_page`: at least 500 impressions, a measured position in the top 20, and CTR below 0.5%.
- `page_one_opportunity`: average position is 4–10 with at least 500 impressions.
- `thin_visible_page`: fewer than 1,200 words and at least 250 impressions.
- `low_engagement_visible_page`: at least 30 sessions and low engagement or scroll rate.
- `general_refresh_review`: no specific reason matched; retained so every row remains reviewable.

The final human output is a ranked, decision-support queue with a score, suggested action, reason codes, and the label used only for evaluation.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from pathlib import Path
import numpy as np
import pandas as pd

def find_repo_root():
    here = Path.cwd().resolve()
    candidates = [here, *here.parents]
    for candidate in candidates:
        if (candidate / "data" / "raw" / "content_refresh_anonymized.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find data/raw/content_refresh_anonymized.csv")

ROOT = find_repo_root()
RAW_PATH = ROOT / "data" / "raw" / "content_refresh_anonymized.csv"
OUTPUT_DIR = ROOT / "work" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(RAW_PATH)

required = [
    "content_id", "client_id", "impressions_90d", "sessions_90d",
    "days_since_last_update", "avg_position", "ctr", "word_count",
    "engagement_rate", "scroll_rate", "trend_direction",
]
missing = sorted(set(required) - set(df.columns))

numeric_columns = [column for column in required if column not in {"content_id", "client_id", "trend_direction"}]
for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce").fillna(0)
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy() if "content_age_days" in df else df[df["impressions_90d"] > 0].copy()
df = df.drop_duplicates("content_id").reset_index(drop=True)
df["is_declining_label"] = df["trend_direction"].astype(str).str.lower().eq("down").astype(int)

print(f"Loaded {len(df):,} unique visible content items")
print(f"Observed declining-label base rate: {df['is_declining_label'].mean():.3f}")

Loaded 30,000 unique visible content items
Observed declining-label base rate: 0.542


## 2. Build the ranked queue (writes the CSV)

The score is deliberately hand-written: 40% visibility, 30% freshness risk, 20% ranking opportunity, and 10% content-depth gap. Percentile ranks make the components comparable without fitting weights to the label. `trend_direction` and `trend_pct` are excluded from the score.

In [3]:
def percentile_rank(series):
    return series.rank(method="average", pct=True).fillna(0)

def reason_codes(row):
    reasons = []
    if row["days_since_last_update"] >= 180 and row["impressions_90d"] >= 500:
        reasons.append("stale_visible_page")
    if row["impressions_90d"] >= 500 and 0 < row["avg_position"] <= 20 and row["ctr"] < 0.5:
        reasons.append("low_ctr_visible_page")
    if row["impressions_90d"] >= 500 and 3 < row["avg_position"] <= 10:
        reasons.append("page_one_opportunity")
    if 0 < row["word_count"] < 1200 and row["impressions_90d"] >= 250:
        reasons.append("thin_visible_page")
    if row["sessions_90d"] >= 30 and (
        0 < row["engagement_rate"] < 30 or 0 < row["scroll_rate"] < 30
    ):
        reasons.append("low_engagement_visible_page")
    return reasons or ["general_refresh_review"]

def suggested_action(reasons):
    reasons = set(reasons)
    if "thin_visible_page" in reasons:
        return "expand_and_refresh"
    if "low_ctr_visible_page" in reasons:
        return "refresh_and_review_ctr"
    if "stale_visible_page" in reasons or "page_one_opportunity" in reasons:
        return "refresh"
    return "monitor"

df["visibility_score"] = percentile_rank(np.log1p(df["impressions_90d"]))
df["freshness_risk_score"] = percentile_rank(df["days_since_last_update"])
position = df["avg_position"].clip(lower=1, upper=50)
df["position_opportunity_score"] = (
    (1 - (position - 1) / 49) * df["visibility_score"] * (df["avg_position"] > 0)
)
df["depth_gap_score"] = (1 - percentile_rank(df["word_count"])) * df["visibility_score"]
df["baseline_refresh_score"] = (
    0.40 * df["visibility_score"]
    + 0.30 * df["freshness_risk_score"]
    + 0.20 * df["position_opportunity_score"]
    + 0.10 * df["depth_gap_score"]
).clip(0, 1)
df["reason_code_list"] = df.apply(reason_codes, axis=1)
df["reason_codes"] = df["reason_code_list"].str.join("|")
df["suggested_action"] = df["reason_code_list"].apply(suggested_action)
df["baseline_rank"] = df["baseline_refresh_score"].rank(method="first", ascending=False).astype(int)

queue_columns = [
    "content_id", "client_id", "baseline_rank", "baseline_refresh_score",
    "visibility_score", "freshness_risk_score", "position_opportunity_score",
    "depth_gap_score", "reason_codes", "suggested_action", "is_declining_label",
    "impressions_90d", "clicks_90d", "sessions_90d", "avg_position", "ctr",
    "engagement_rate", "scroll_rate", "content_age_days", "days_since_last_update",
    "word_count", "trend_direction",
]
queue = df[queue_columns].sort_values("baseline_rank").reset_index(drop=True)
queue_path = OUTPUT_DIR / "baseline_action_score.csv"
queue.to_csv(queue_path, index=False)
print(f"Wrote {len(queue):,} ranked rows to {queue_path}")
print(queue[["baseline_rank", "baseline_refresh_score", "reason_codes", "suggested_action"]].head(5).to_string(index=False))

Wrote 30,000 ranked rows to /content/flyrank-ml-internship-starter/work/outputs/baseline_action_score.csv
 baseline_rank  baseline_refresh_score                                                          reason_codes       suggested_action
             1                0.935795                                           low_engagement_visible_page                monitor
             2                0.930031                                           low_engagement_visible_page                monitor
             3                0.929496 low_ctr_visible_page|page_one_opportunity|low_engagement_visible_page refresh_and_review_ctr
             4                0.929424                                           low_engagement_visible_page                monitor
             5                0.929247                      low_ctr_visible_page|low_engagement_visible_page refresh_and_review_ctr


## 3. Top-20 review

The table below is the required human-review checklist. Confidence is a heuristic note, not a calibrated probability. A pick is more actionable when it has multiple reasons and enough impressions; it is weaker when visibility is small or measurement fields are missing.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
review = queue.head(20).copy()
review["confidence_note"] = np.where(
    review["impressions_90d"] >= 1000,
    "higher confidence: enough impressions for directional review",
    "lower confidence: small sample; verify before acting",
)
review["what_would_make_it_wrong"] = np.where(
    review["impressions_90d"] < 1000,
    "an extra click or sparse tracking may explain the rate",
    "the metric window may not represent current search demand",
)
review_columns = [
    "baseline_rank", "content_id", "suggested_action", "reason_codes",
    "confidence_note", "what_would_make_it_wrong", "impressions_90d",
    "avg_position", "ctr", "days_since_last_update",
]
top20_review = review[review_columns]
display(top20_review)
top20_review.to_csv(OUTPUT_DIR / "baseline_top20_review.csv", index=False)
print("Saved the top-20 review checklist.")

,baseline_rank,content_id,suggested_action,reason_codes,confidence_note,what_would_make_it_wrong,impressions_90d,avg_position,ctr,days_since_last_update
0,1,content_9532f197bbc8,monitor,low_engagement_visible_page,higher confidence: enough impressions for dire...,the metric window may not represent current se...,309192,2.0,0.87,104
1,2,content_4d1fe5b32dc2,monitor,low_engagement_visible_page,higher confidence: enough impressions for dire...,the metric window may not represent current se...,97999,2.5,0.52,104
2,3,content_3430a8b94511,refresh_and_review_ctr,low_ctr_visible_page|page_one_opportunity|low_...,higher confidence: enough impressions for dire...,the metric window may not represent current se...,152617,3.3,0.29,104
3,4,content_07f2e7a6f38a,monitor,low_engagement_visible_page,higher confidence: enough impressions for dire...,the metric window may not represent current se...,101078,2.7,0.85,104
4,5,content_e5ae436f9a16,refresh_and_review_ctr,low_ctr_visible_page|low_engagement_visible_page,higher confidence: enough impressions for dire...,the metric window may not represent current se...,117741,3.0,0.45,104
5,6,content_cbd93118300b,refresh_and_review_ctr,low_ctr_visible_page|page_one_opportunity|low_...,higher confidence: enough impressions for dire...,the metric window may not represent current se...,145292,3.3,0.46,104
6,7,content_9c195417f6ef,monitor,low_engagement_visible_page,higher confidence: enough impressions for dire...,the metric window may not represent current se...,79146,2.5,0.73,104
7,8,content_ba2acb4ebd04,refresh,page_one_opportunity|low_engagement_visible_page,higher confidence: enough impressions for dire...,the metric window may not represent current se...,142072,3.6,0.83,104
8,9,content_79b25654070a,refresh_and_review_ctr,low_ctr_visible_page|page_one_opportunity|low_...,higher confidence: enough impressions for dire...,the metric window may not represent current se...,148737,3.7,0.48,104
9,10,content_adddad39251c,refresh,page_one_opportunity|low_engagement_visible_page,higher confidence: enough impressions for dire...,the metric window may not represent current se...,129239,3.6,0.55,104


Saved the top-20 review checklist.


## 4. Weak picks + leakage check


A transparent baseline should expose weak picks rather than hide them. We flag top-ranked rows with fewer than 100 impressions or with no measurable ranking position for manual verification. The score must not use the target or product-decision fields.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
weak_picks = queue.head(20).loc[
    (queue.head(20)["impressions_90d"] < 100)
    | (queue.head(20)["avg_position"] <= 0)
]
print(f"Weak top-20 picks needing manual verification: {len(weak_picks)}")
display(weak_picks[["baseline_rank", "content_id", "impressions_90d", "avg_position", "reason_codes"]])

score_inputs = {
    "impressions_90d", "days_since_last_update", "avg_position", "ctr",
    "word_count", "sessions_90d", "engagement_rate", "scroll_rate",
}
forbidden_inputs = {"trend_direction", "trend_pct", "is_declining_label", "provider_used", "model_used"}
assert score_inputs.isdisjoint(forbidden_inputs)
assert len(queue) == len(df)
assert queue["baseline_rank"].is_unique
assert queue_path.exists()

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(labels)[order[:k]].mean())

for k in [10, 20, 50, 100]:
    measured = precision_at_k(df["baseline_refresh_score"], df["is_declining_label"], k)
    print(f"Precision@{k}: {measured:.3f} | base rate: {df['is_declining_label'].mean():.3f}")

print("Leakage check passed: the score uses current-window/content fields only;")
print("trend_direction/trend_pct are retained for evaluation, never used as score inputs.")

Weak top-20 picks needing manual verification: 0


,baseline_rank,content_id,impressions_90d,avg_position,reason_codes


Precision@10: 0.200 | base rate: 0.542
Precision@20: 0.350 | base rate: 0.542
Precision@50: 0.320 | base rate: 0.542
Precision@100: 0.360 | base rate: 0.542
Leakage check passed: the score uses current-window/content fields only;
trend_direction/trend_pct are retained for evaluation, never used as score inputs.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.